# 05.02 — Profile vs. Definition

This notebook re-exercises the original profile-vs-definition comparison
using the renamed API introduced in E27:

| Old (removed) | New |
|---|---|
| `compare(profile, graph_definition)` | `compare_profile_to_definition(profile, graph_definition)` |

The semantics and emitted codes are **identical** — only the function name
changed. `compare_profile_to_definition` still uses `standard_rules()` which
asks: *does the observed profile satisfy the declared definition?*

All three comparison functions live in `orthograph.comparison.engine`:

```python
from orthograph.comparison.engine import (
    compare_profile_to_definition,  # satisfaction check (ERROR/WARNING/INFO)
    compare_profiles,               # symmetric diff of two profiles (INFO only)
    compare_definitions,            # symmetric diff of two definitions (INFO only)
)
```

This notebook:
1. Builds a `GraphDefinition` and two `GraphProfile` objects (one valid, one with issues).
2. Calls `compare_profile_to_definition` and renders the `ValidationResult`.
3. Shows how `is_valid` and `errors` work for actionable feedback.

In [ ]:
from orthograph.comparison.engine import compare_profile_to_definition
from orthograph.diagnostics.result import GraphValidationError

## 1. Declare the graph model

In [ ]:
from shared.filmography import FILMOGRAPHY_MODEL


graph_definition = FILMOGRAPHY_MODEL
print("Model:", graph_definition.name, "| nodes:", sorted(graph_definition.node_labels))

## 2. Get a profile that satisfies the definition

All required properties present, correct types, endpoints and cardinality within bounds.

In [ ]:
from shared.profiles import FILMOGRAPHY_PROFILE


valid_profile = FILMOGRAPHY_PROFILE

result_ok = compare_profile_to_definition(valid_profile, graph_definition)

assert result_ok.is_valid
assert len(result_ok.issues) == 0
assert len(result_ok.errors) == 0
print(
    f"Valid profile  — is_valid: {result_ok.is_valid}, issues: {len(result_ok.issues)}, errors: {len(result_ok.errors)}"
)

## 3. Get a profile with validation issues

Deliberate problems:
- `released` property missing from Movie (required → `MISSING_PROPERTY`).
- ACTED_IN `role` property has type `Long` instead of `String` (`PROPERTY_TYPE_MISMATCH`).
- ACTED_IN cardinality `min=0` violates `1..*` (`CARDINALITY_VIOLATION`).
- Unexpected `Genre` node label in profile (`UNEXPECTED_NODE_LABEL`).

In [ ]:
from shared.profiles import FILMOGRAPHY_PROFILE_INVALID


invalid_profile = FILMOGRAPHY_PROFILE_INVALID

result_bad = compare_profile_to_definition(invalid_profile, graph_definition)

assert result_bad.is_valid is False
assert len(result_bad.issues) == 8
assert len(result_bad.errors) == 4

print(
    f"Invalid profile — is_valid: {result_bad.is_valid}, issues: {len(result_bad.issues)}, errors: {len(result_bad.errors)}"
)

## 4. Render all issues

In [ ]:
for issue in result_bad.issues:
    print(f"[{issue.severity.value.upper():7}] {issue.code:<30} {issue.entity_id}")
    print(f"          {issue.message}")
    if issue.context:
        print(f"          context: {issue.context}")
    print()

## 5. Raise on errors (optional)

`ValidationResult.raise_on_errors()` converts collected errors into a
`GraphValidationError` exception — convenient for CI pipelines.

In [ ]:
try:
    result_bad.raise_on_errors()
except GraphValidationError as exc:
    print("Caught GraphValidationError:")
    for issue in exc.issues:
        print(f"  [{issue.severity.value.upper()}] {issue.code}: {issue.message}")

## 6. All three comparison functions at a glance

In [ ]:
print("compare_profile_to_definition — satisfaction check")
print("  Question: does the observed profile satisfy the declared constraints?")
print("  Default rules: standard_rules() (ERROR / WARNING / INFO)")
print("  is_valid = True only when zero ERRORs")
print()
print("compare_profiles — symmetric diff of two profiles")
print("  Question: what structural differences exist between two observed snapshots?")
print("  Default rules: diff_rules() (INFO only)")
print("  is_valid = always True")
print()
print("compare_definitions — symmetric diff of two definitions")
print("  Question: what changed between two declared schemas?")
print("  Default rules: diff_rules() (INFO only)")
print("  is_valid = always True")